In [1]:
from deepchem.feat import MolGraphConvFeaturizer
from data.preprocessing.pipeline import Pipeline
pipeline = Pipeline(download=False)

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
c:\Users\Silve\Jupyter\Jupyter projects\Working area\thortex-hackaton\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'dgl'
Skipped loading modules w

In [2]:
import numpy as np
import pandas as pd
import random
import torch
from sklearn.model_selection import train_test_split
from deepchem.models import DMPNNModel
from deepchem.data import NumpyDataset
from tqdm import tqdm
import os
from utils.paths import WEIGHTS_DIR

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def train_model(
    pipeline: Pipeline,
    split_ratio: float = 0.2,
    random_state: int = 42,
    epochs: int = 30,
    batch_size: int = 32,
    learning_rate: float = 0.001,
    model_dir: str = "dmpnn_model"
):
    """Обучение DMPNN модели и сохранение её на диск"""
    set_seed(random_state)

    model_dir=os.path.join(WEIGHTS_DIR, model_dir)
    os.makedirs(model_dir, exist_ok=True)

    # Загружаем графы
    (train_graphs, train_targets), _ = pipeline.get_processed_data()

    # Извлекаем метки
    train_labels = train_targets[:, 1].astype(np.int32)

    # Разделение на train / val
    train_idx, val_idx = train_test_split(
        np.arange(len(train_graphs)),
        test_size=split_ratio,
        random_state=random_state,
        stratify=train_labels
    )

    train_graphs_split = [train_graphs[i] for i in train_idx]
    val_graphs_split   = [train_graphs[i] for i in val_idx]
    train_labels_split = train_labels[train_idx]
    val_labels_split   = train_labels[val_idx]

    # Создание датасетов
    train_dataset = NumpyDataset(
        X=np.array(train_graphs_split, dtype=object),
        y=train_labels_split.reshape(-1, 1)
    )
    val_dataset = NumpyDataset(
        X=np.array(val_graphs_split, dtype=object),
        y=val_labels_split.reshape(-1, 1)
    )

    # Инициализация модели
    n_classes = 8

    model = DMPNNModel(
        n_tasks=1,
        n_classes=n_classes,
        mode='classification',
        batch_size=batch_size,
        learning_rate=learning_rate,
        epochs=epochs,
        model_dir=model_dir
    )

    print("[INFO] Training DMPNN...")

    for epoch in tqdm(range(epochs), desc="Epochs"):
        model.fit(train_dataset, nb_epoch=1)

    print(f"[INFO] Training DMPNN for {n_classes} classes...")

    val_preds = model.predict(val_dataset)
    val_pred_labels = np.argmax(val_preds, axis=1)
    val_acc = np.mean(val_pred_labels == val_labels_split)

    print(f"[INFO] Validation Accuracy: {val_acc:.4f}")

    # Сохранение
    model.save_checkpoint(model_dir=model_dir)
    print(f"[INFO] Model saved to: {model_dir}")

In [3]:
def predict_model(
    pipeline: Pipeline,
    model_dir: str = "dmpnn_model",
    output_file: str = "submission.csv"
):
    """Загрузка сохранённой DMPNN модели и предсказание на тесте"""

    model_dir=os.path.join(WEIGHTS_DIR, model_dir)
    _, (test_graphs, test_targets) = pipeline.get_processed_data()

    # Добавить global_features для DMPNN
    test_graphs = test_graphs

    test_mol_ids = test_targets[:, 0]

    test_dataset = NumpyDataset(X=np.array(test_graphs, dtype=object))

    # Загрузка модели
    n_classes = 8
    model = DMPNNModel(
        n_tasks=1,
        n_classes=n_classes,
        mode='classification',
        model_dir=model_dir
    )
    model.restore(model_dir=model_dir)

    print(f"[INFO] Loaded model from {model_dir}")

    # Предсказание
    test_preds = model.predict(test_dataset)
    test_pred_labels = np.argmax(test_preds, axis=1)

    # Сохранение результатов
    submission = pd.DataFrame({
        "taste_cluster": test_pred_labels.astype(int)
    })

    submission.to_csv(output_file, index=False)
    print(f"[INFO] Predictions saved to {output_file}, shape={submission.shape}")


In [4]:
train_model(
    pipeline=pipeline,
    model_dir="mpnn_model",
    epochs=15,
    learning_rate=0.005,
    batch_size=64,
    split_ratio=0.3,
    random_state=42
)

[INFO] Training DMPNN...


Epochs:   0%|          | 0/15 [00:00<?, ?it/s]c:\Users\Silve\Jupyter\Jupyter projects\Working area\thortex-hackaton\.venv\Lib\site-packages\deepchem\models\torch_models\torch_model.py:475: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  avg_loss = float(avg_loss) / averaged_batches
Epochs: 100%|██████████| 15/15 [00:41<00:00,  2.77s/it]


[INFO] Training DMPNN for 8 classes...
[INFO] Validation Accuracy: 0.4081
[INFO] Model saved to: C:\Users\Silve\Jupyter\Jupyter projects\Working area\thortex-hackaton\weights\mpnn_model


In [5]:
predict_model(
    pipeline=pipeline,
    model_dir="mpnn_model",
    output_file="submission.csv"
)

[INFO] Loaded model from C:\Users\Silve\Jupyter\Jupyter projects\Working area\thortex-hackaton\weights\mpnn_model
[INFO] Predictions saved to submission.csv, shape=(888, 1)
